In [1]:
import xarray as xr
import numpy as np
import glob

In [ ]:
VARIABLES = [
    'TMQ', 'U850', 'V850', 'UBOT', 'VBOT',
    'QREFHT', 'PS', 'PSL', 'T200', 'T500',
    'PRECT', 'TS', 'TREFHT', 'Z1000', 'Z200', 'ZBOT'
]

files = sorted(glob.glob("../../climatenet/train/*.nc"))

n = {v: 0    for v in VARIABLES}  # nombre de pixels vus
mean = {v: 0.0  for v in VARIABLES}  # moyenne courante
M2 = {v: 0.0  for v in VARIABLES}  # somme des carrés des écarts (pour std)

print(len(M2))
print(len(files))

16
398


In [9]:
for f in files:
    ds = xr.open_dataset(f)
    for v in VARIABLES:
        data = ds[v].values.flatten().astype(np.float64)
        batch_n = len(data)
        batch_mean = data.mean()
        batch_M2  = ((data - batch_mean) ** 2).sum()
        delta = batch_mean - mean[v]
        new_n = n[v] + batch_n
        mean[v] = (n[v] * mean[v] + batch_n * batch_mean) / new_n
        M2[v] += batch_M2 + delta**2 * n[v] * batch_n / new_n
        n[v] = new_n
    ds.close()

In [15]:
for v in VARIABLES:
    std = np.sqrt(M2[v] / n[v])
    print(f'"{v}": {{"mean": {mean[v]:.10e}, "std": {std:.10e}}}')

"TMQ": {"mean": 1.9218493403e+01, "std": 1.5817276818e+01}
"U850": {"mean": 1.5530236523e+00, "std": 8.2976213423e+00}
"V850": {"mean": 2.5413171355e-01, "std": 6.2316312967e+00}
"UBOT": {"mean": 1.2487871773e-01, "std": 6.6532873420e+00}
"VBOT": {"mean": 3.1541637515e-01, "std": 5.7841890396e+00}
"QREFHT": {"mean": 7.7877782256e-03, "std": 6.2152877903e-03}
"PS": {"mean": 9.6571613897e+04, "std": 9.7001007858e+03}
"PSL": {"mean": 1.0081407257e+05, "std": 1.4612256421e+03}
"T200": {"mean": 2.1320908332e+02, "std": 7.8897894061e+00}
"T500": {"mean": 2.5303822241e+02, "std": 1.2825335782e+01}
"PRECT": {"mean": 2.9458238675e-08, "std": 1.5563764198e-07}
"TS": {"mean": 2.7871147711e+02, "std": 2.3682519490e+01}
"TREFHT": {"mean": 2.7842122233e+02, "std": 2.2511935251e+01}
"Z1000": {"mean": 4.7417278496e+02, "std": 8.3280818810e+02}
"Z200": {"mean": 1.1736104621e+04, "std": 6.3325807434e+02}
"ZBOT": {"mean": 6.1311490174e+01, "std": 4.9094973858e+00}
